# TP53 Project Pipeline Runner

Use this notebook to run the project end to end from the repository root. The scripts do the heavy lifting and write reproducible outputs to `data/processed/`, `reports/tables/`, `reports/figures/`, and `models/`.

In [2]:
from pathlib import Path
import os

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
ROOT

PosixPath('/Users/v_angelov/GroupProj/TP53-MUTATIONS')

## 1. Download DepMap/CCLE Data

Skip this cell if `data/raw/` already contains the DepMap files.

In [3]:
!python3 scripts/00_download_data.py

Downloaded expression: data/raw/OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv
Downloaded mutations:  data/raw/OmicsSomaticMutations.csv
Downloaded metadata:   data/raw/Model.csv


## 2. Build the Matched TP53 Dataset

In [4]:
!python3 scripts/01_build_dataset.py

Expression shape: (1719, 19215)
tp53_mutant
Mutant    997
WT        722
Name: count, dtype: int64
Wrote data/processed/expression_matched.csv.gz
Wrote data/processed/tp53_labels.csv
Wrote data/processed/sample_metadata.csv


## 3. Generate EDA Outputs

In [5]:
!python3 scripts/02_eda.py

              metric         value
0            samples   1719.000000
1              genes  19215.000000
2     missing_values      0.000000
3    mean_expression      2.670069
4  median_expression      2.415231


## 4. Train Binary TP53 Mutation Models

In [6]:
!python3 scripts/03_train_binary.py

Minimum samples required per stratum: 7
Samples before filtering: 1,719
Samples after filtering:  964
Samples removed:          755
Cancer types retained:    20
Strata retained:          49
top_3000_variable / majority: val ROC-AUC=0.500, F1=0.728
top_3000_variable / logistic_l2: val ROC-AUC=0.885, F1=0.855
/Users/v_angelov/.pyenv/versions/3.12.7/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
top_3000_variable / elastic_net_logistic: val ROC-AUC=0.889, F1=0.826
top_3000_variable / linear_svm: val ROC-AUC=0.886, F1=0.857
top_3000_variable / random_forest: val ROC-AUC=0.854, F1=0.814
top_3000_variable / extra_trees: val ROC-AUC=0.852, F1=0.802
top_3000_varia

## 5. Train Mutation-Type Models

In [10]:
!python3 scripts/04_train_multiclass.py

Minimum samples required per cancer type: 7
Samples before cancer-type filter: 1,719
Samples after cancer-type filter:  1,600
Cancer types retained: 42
Samples after mutation-class filter: 1,583
Classes retained: ['Frameshift', 'Missense', 'Nonsense', 'Other', 'WT']
top_3000_variable / majority: val macro-F1=0.118
top_3000_variable / logistic_l2: val macro-F1=0.313
top_3000_variable / elastic_net_logistic: val macro-F1=0.359
top_3000_variable / linear_svm: val macro-F1=0.334
top_3000_variable / random_forest: val macro-F1=0.264
top_3000_variable / extra_trees: val macro-F1=0.284
top_3000_variable / hist_gradient_boosting: val macro-F1=0.279
top_3000_variable / mlp: val macro-F1=0.343
tp53_targets / majority: val macro-F1=0.118
tp53_targets / logistic_l2: val macro-F1=0.369
tp53_targets / elastic_net_logistic: val macro-F1=0.384
tp53_targets / linear_svm: val macro-F1=0.387
tp53_targets / random_forest: val macro-F1=0.369
tp53_targets / extra_trees: val macro-F1=0.358
tp53_targets / his

## 6. Biological Interpretation

In [11]:
!python3 scripts/05_interpret.py

                                        gene_set  ...                                              genes
0  TP53 direct targets in top differential genes  ...  AEN;BAX;BBC3;CDKN1A;DDB2;DRAM1;FAS;FDXR;MDM2;P...
1    p53 pathway genes in top differential genes  ...  AEN;BAX;BBC3;CDKN1A;DDB2;DRAM1;FAS;FDXR;MDM2;P...
2                            TP53 direct targets  ...  AEN;APAF1;BAX;BBC3;BTG2;CASP1;CD82;CDKN1A;DDB2...
3                                    p53 pathway  ...  AEN;APAF1;ATM;ATR;BAK1;BAX;BBC3;BCL2;BCL2L1;BI...

[4 rows x 3 columns]
